# BEAM Stage 1: CG Model Evaluation

Complete evaluation framework for CG models on real protein data.

---

## Multi-System Support

This notebook supports evaluation across multiple protein systems and CG models:

### **System 1: Pertactin**
- **Protein:** Pertactin (106 residues, β-helical autotransporter)
- **Biological Process:** Folding/unfolding
- **CG Models:** Upside (backbone N, CA, C)
- **AA Reference:** CHARMM all-atom simulation
- **Trajectory Length:** 2000 frames each

### **System 2: ADK (Adenylate Kinase)**
- **Protein:** Adenylate kinase (larger protein)
- **Biological Process:** Open/closed conformational transition (relevant for ligand binding)
- **CG Models:** Upside (backbone N, CA, C), SIRAH
- **AA Reference:** CHARMM all-atom simulation
- **Trajectory Length:** 2000 frames each

---

## Evaluation Dimensions

1. **Distribution-based:** RDF + CV distributions (KS test, KL divergence, JS divergence, overlap coefficient)
2. **Dynamics-based:** Autocorrelation times, decorrelation rates, speedup factors
3. **Sampling efficiency:** Effective sample size (ESS), basin discovery, conformational variance

---

## Key Innovation: In-Support Analysis

This evaluation framework includes an **in-support analysis** method that:
- Compares CG vs AA distributions **only within AA's sampled region** (5th-95th percentile)
- Distinguishes between:
  - **Force field errors** (CG inaccurate even in overlap region)
  - **Enhanced sampling** (CG accurate in overlap, but explores new regions)
- Provides clear guidance for CG model selection in downstream workflows

---

## Usage

**To switch systems/models:** Edit the configuration in Cell 2:

```python
SYSTEM = 'pertactin'  # or 'adk'
CG_MODEL = 'upside'   # or 'sirah'
```

Then run all cells!


## 0. Configuration

**Customize your alignment and selection here**

In [ ]:
# ============================
# USER CONFIGURATION
# ============================
import sys
sys.path.insert(0, '..')

# ========== SELECT SYSTEM AND CG MODEL ==========
SYSTEM = 'adk'  # Options: 'pertactin', 'adk'
CG_MODEL = 'sirah'   # Options: 'upside', 'sirah'

print(f"System: {SYSTEM}")
print(f"CG Model: {CG_MODEL}")

# ========== SYSTEM CONFIGURATIONS ==========
SYSTEM_CONFIGS = {
    'pertactin': {
        'data_dir': '../data/pertactin',
        'output_dir': '../results/stage1_evaluation/pertactin',
        'aa_dcd': 'aa.dcd',
        'aa_topology': 'Cside_protein.pdb',
        'aa_reference': 'Cside_protein.pdb',
        'cg_files': {
            'upside': {
                'dcd': 'cg_upside.dcd',
                'topology': 'C_upside.psf',
                'reference': 'C_upside.psf'
            },
        },
        'align_selection': 'name CA and resid 446 to 468',
        'description': 'Pertactin (106 residues, β-helical folding/unfolding)'
    },
    'adk': {
        'data_dir': '../data/adk',
        'output_dir': '../results/stage1_evaluation/adk',
        'aa_dcd': 'aa.dcd',
        'aa_topology': 'adk_protein.psf',
        'aa_reference': 'adk_protein.pdb',
        'cg_files': {
            'upside': {
                'dcd': 'cg_upside.dcd',
                'topology': 'adk_upside.psf',
                'reference': 'adk_upside.psf'
            },
            'sirah': {
                'dcd': 'cg_sirah.dcd',
                'topology': 'cg_sirah.prmtop',
                'reference': 'cg_sirah.pdb'
            }
        },
        'align_selection': 'name CA',
        'description': 'ADK (adenylate kinase, open/closed transition)'
    }
}

# ========== LOAD SELECTED CONFIGURATION ==========
if SYSTEM not in SYSTEM_CONFIGS:
    raise ValueError(f"Unknown system: {SYSTEM}. Available: {list(SYSTEM_CONFIGS.keys())}")

if CG_MODEL not in SYSTEM_CONFIGS[SYSTEM]['cg_files']:
    raise ValueError(f"CG model {CG_MODEL} not configured for {SYSTEM}")

config = SYSTEM_CONFIGS[SYSTEM]
cg_config = config['cg_files'][CG_MODEL]

# ========== FILE PATHS ==========
DATA_DIR = config['data_dir']
OUTPUT_DIR = config['output_dir']

# CG files
CG_DCD = f"{DATA_DIR}/{cg_config['dcd']}"
CG_TOPOLOGY = f"{DATA_DIR}/{cg_config['topology']}"

# AA files
AA_DCD = f"{DATA_DIR}/{config['aa_dcd']}"
AA_TOPOLOGY = f"{DATA_DIR}/{config['aa_topology']}"

# Reference structures for alignment
AA_REFERENCE = config.get('aa_reference', config.get('reference_pdb'))
if not AA_REFERENCE:
    raise ValueError("No AA reference structure specified in config!")
AA_REFERENCE_PATH = f"{DATA_DIR}/{AA_REFERENCE}"

CG_REFERENCE = cg_config.get('reference')
if CG_REFERENCE:
    CG_REFERENCE_PATH = f"{DATA_DIR}/{CG_REFERENCE}"
else:
    CG_REFERENCE_PATH = CG_TOPOLOGY

# For backward compatibility with rest of notebook
REFERENCE_PDB = AA_REFERENCE_PATH

# Alignment selection
ALIGN_SELECTION = config['align_selection']

# ========== RDF PARAMETERS ==========
RDF_PARAMS = {
    'r_max': 15,      # Maximum distance (angstrom)
    'bins': 100,      # Number of bins
    'box_size': None  # Set to box size if using PBC, None otherwise
}

# ========== SAMPLING PARAMETERS ==========
N_CLUSTERS = 10  # Number of clusters for basin discovery

# ========== SUMMARY ==========
print("\n" + "="*70)
print(f"SYSTEM: {SYSTEM.upper()}")
print(f"CG MODEL: {CG_MODEL.upper()}")
print("="*70)
print(f"Description: {config['description']}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nFiles:")
print(f"  AA trajectory: {config['aa_dcd']}")
print(f"  AA topology: {config['aa_topology']}")
print(f"  AA reference: {AA_REFERENCE}")
print(f"  CG trajectory: {cg_config['dcd']}")
print(f"  CG topology: {cg_config['topology']}")
print(f"  CG reference: {CG_REFERENCE if CG_REFERENCE else '(using topology)'}")
print(f"\nParameters:")
print(f"  Alignment selection: {ALIGN_SELECTION}")
print(f"  RDF max distance: {RDF_PARAMS['r_max']} Å")
print(f"  Basin discovery clusters: {N_CLUSTERS}")
print("="*70)

In [ ]:
# ============================
# RDF PARAMETERS
# ============================
RDF_PARAMS = {
    'r_max': 15,      # Maximum distance (angstrom)
    'bins': 100,       # Number of bins
    'box_size': None   # Set to box size if using PBC, None otherwise
}

# ============================
# SAMPLING PARAMETERS
# ============================
N_CLUSTERS = 10  # Number of clusters for basin discovery

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import align
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# BEAM imports
from beam import CGEvaluator
from beam.cg_models import get_cg_model_config

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Create output directory
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("✓ Imports successful")
print(f"✓ Output directory created: {OUTPUT_DIR}")

## 2. Load Trajectories

In [ ]:
print("\n" + "="*70)
print("LOADING TRAJECTORIES")
print("="*70)

# Load CG trajectory
print("\n[1/3] Loading CG trajectory...")
u_cg = mda.Universe(CG_TOPOLOGY, CG_DCD)
print(f"  ✓ CG loaded")
print(f"    Atoms: {u_cg.atoms.n_atoms}")
print(f"    Residues: {u_cg.atoms.n_residues}")
print(f"    Frames: {len(u_cg.trajectory)}")

# Load AA trajectory
print("\n[2/3] Loading AA trajectory...")
u_aa = mda.Universe(AA_TOPOLOGY, AA_DCD)
print(f"  ✓ AA loaded")
print(f"    Atoms: {u_aa.atoms.n_atoms}")
print(f"    Residues: {u_aa.atoms.n_residues}")
print(f"    Frames: {len(u_aa.trajectory)}")

# Load reference structure
print("\n[3/3] Loading reference structure...")
u_ref = mda.Universe(REFERENCE_PDB)
print(f"  ✓ Reference loaded")
print(f"    Atoms: {u_ref.atoms.n_atoms}")

print("\n✓ All trajectories loaded successfully")

## 3. Align Trajectories

Remove overall translation and rotation by aligning to reference structure.

In [ ]:
# ============================
# ALIGNMENT
# ============================
import MDAnalysis.analysis.align as align
from beam.cg_models import get_cg_model_config

print("\n" + "="*70)
print("ALIGNING TRAJECTORIES TO REFERENCE")
print("="*70)

# Get CG model configuration
cg_model_config = get_cg_model_config(CG_MODEL)

# Define alignment selections
aa_align_sel = ALIGN_SELECTION  # From system config
cg_align_sel = cg_model_config['align_selection']  # From CG model config

print(f"\nAlignment selections:")
print(f"  AA:  {aa_align_sel}")
print(f"  CG:  {cg_align_sel}")

print(f"\nReference structures:")
print(f"  AA:  {AA_REFERENCE_PATH}")
print(f"  CG:  {CG_REFERENCE_PATH}")

# ========== ALIGN AA TRAJECTORY ==========
print(f"\n[1/2] Aligning AA trajectory to AA reference...")

# Load AA reference
u_ref_aa = mda.Universe(AA_REFERENCE_PATH)
aa_ref_atoms = u_ref_aa.select_atoms(aa_align_sel)
aa_align_atoms = u_aa.select_atoms(aa_align_sel)

print(f"  Reference atoms: {aa_ref_atoms.n_atoms}")
print(f"  Trajectory atoms: {aa_align_atoms.n_atoms}")

if aa_align_atoms.n_atoms != aa_ref_atoms.n_atoms:
    raise ValueError(
        f"AA trajectory and reference have different number of alignment atoms! "
        f"Trajectory: {aa_align_atoms.n_atoms}, Reference: {aa_ref_atoms.n_atoms}"
    )

if aa_align_atoms.n_atoms == 0:
    raise ValueError(f"No AA atoms selected with '{aa_align_sel}'")

# Align AA trajectory
align.AlignTraj(
    u_aa,
    u_ref_aa,
    select=aa_align_sel,
    in_memory=True,
    match_atoms=False
).run()
print("  ✓ AA alignment complete")

# ========== ALIGN CG TRAJECTORY ==========
print(f"\n[2/2] Aligning CG trajectory to CG reference...")

# Load CG reference
try:
    u_ref_cg = mda.Universe(CG_REFERENCE_PATH)
    print(f"  Loaded CG reference: {CG_REFERENCE_PATH}")
except Exception as e:
    # If CG_REFERENCE_PATH is a topology file without coordinates,
    # use the first frame of the trajectory
    print(f"  Note: {CG_REFERENCE_PATH} has no coordinates, using trajectory's first frame")
    u_ref_cg = mda.Universe(CG_TOPOLOGY, CG_DCD)
    u_ref_cg.trajectory[0]

cg_ref_atoms = u_ref_cg.select_atoms(cg_align_sel)
cg_align_atoms = u_cg.select_atoms(cg_align_sel)

print(f"  Reference atoms: {cg_ref_atoms.n_atoms}")
print(f"  Trajectory atoms: {cg_align_atoms.n_atoms}")

if cg_align_atoms.n_atoms != cg_ref_atoms.n_atoms:
    raise ValueError(
        f"CG trajectory and reference have different number of alignment atoms! "
        f"Trajectory: {cg_align_atoms.n_atoms}, Reference: {cg_ref_atoms.n_atoms}"
    )

if cg_align_atoms.n_atoms == 0:
    raise ValueError(
        f"No CG atoms selected with '{cg_align_sel}'! "
        f"Check CG model configuration in beam/cg_models.py"
    )

# Align CG trajectory
align.AlignTraj(
    u_cg,
    u_ref_cg,
    select=cg_align_sel,
    in_memory=True,
    match_atoms=False
).run()
print("  ✓ CG alignment complete")

print("\n" + "="*70)
print("✓ ALIGNMENT COMPLETE")
print("="*70)
print(f"\nAligned trajectory information:")
print(f"  AA: {len(u_aa.trajectory)} frames, {aa_align_atoms.n_atoms} alignment atoms ({aa_align_sel})")
print(f"  CG: {len(u_cg.trajectory)} frames, {cg_align_atoms.n_atoms} alignment atoms ({cg_align_sel})")
print(f"\nNote: AA and CG use different alignment selections and references.")
print(f"      This is expected when comparing different force fields.")

## 4. Extract Features for Evaluation

Extract xyz coordinates as features. For Upside, this includes backbone atoms (N, CA, C).

In [ ]:
# ============================
# FEATURE EXTRACTION
# ============================
from beam.cg_models import get_cg_model_config

print("\n" + "="*70)
print("FEATURE EXTRACTION")
print("="*70)

# Get CG model configuration
cg_config = get_cg_model_config(CG_MODEL)
print(f"\nCG Model: {CG_MODEL}")
print(f"  Description: {cg_config['description']}")
print(f"  Feature selection: {cg_config['feature_selection']}")
print(f"  Expected atoms/residue: {cg_config['expected_atoms_per_residue']}")

# Extract CG features
print("\n[1/2] Extracting CG features...")
cg_feature_atoms = u_cg.select_atoms(cg_config['feature_selection'])
print(f"  Selected {cg_feature_atoms.n_atoms} atoms")

# Only print expected count if it's defined
if cg_config['expected_atoms_per_residue'] is not None:
    expected = u_cg.atoms.n_residues * cg_config['expected_atoms_per_residue']
    print(f"  Expected: {expected} atoms")
    if cg_feature_atoms.n_atoms != expected:
        print(f"  ⚠ Warning: Selected atoms ({cg_feature_atoms.n_atoms}) != expected ({expected})")
else:
    print(f"  Note: Variable atoms per residue (CG model dependent)")

cg_features = []
for ts in u_cg.trajectory:
    cg_features.append(cg_feature_atoms.positions.flatten())
cg_features = np.array(cg_features)

print(f"  ✓ CG features extracted")
print(f"    Shape: {cg_features.shape}")
print(f"    ({cg_features.shape[0]} frames × {cg_features.shape[1]} features)")

# Extract AA features (same selection string for comparison)
print("\n[2/2] Extracting AA features...")

# For AA, use the standard CA selection for comparison
# (CG feature selection might not work on AA)
aa_selection = ALIGN_SELECTION  # Use the same alignment selection
aa_feature_atoms = u_aa.select_atoms(aa_selection)
print(f"  Selected {aa_feature_atoms.n_atoms} atoms (using: {aa_selection})")

aa_features = []
for ts in u_aa.trajectory:
    aa_features.append(aa_feature_atoms.positions.flatten())
aa_features = np.array(aa_features)

print(f"  ✓ AA features extracted")
print(f"    Shape: {aa_features.shape}")

# Note: CG and AA feature dimensions may differ
print(f"\n✓ Feature extraction complete")
print(f"  CG features: {cg_features.shape[1]} dimensions")
print(f"  AA features: {aa_features.shape[1]} dimensions")

if cg_features.shape[1] != aa_features.shape[1]:
    print(f"\nℹ Note: CG and AA have different feature dimensions.")
    print(f"  This is expected when using different force fields.")
    print(f"  Evaluation will use collective variables (CVs) for comparison.")

print("="*70)

## 5. Extract Positions for RDF

For RDF calculation, we use backbone atoms only (mapped to same resolution for CG and AA).

In [ ]:
print("\n" + "="*70)
print("EXTRACTING POSITIONS FOR RDF")
print("="*70)
 
# For RDF, use CA atoms (or equivalent backbone beads)
aa_rdf_sel = 'name CA'
cg_rdf_sel = cg_model_config['align_selection']  # Use same as alignment
 
print(f"\nRDF selections:")
print(f"  AA:  {aa_rdf_sel}")
print(f"  CG:  {cg_rdf_sel}")
 
# Extract AA CA positions
print("\n[1/2] Extracting AA CA positions...")
aa_ca_atoms = u_aa.select_atoms(aa_rdf_sel)
print(f"  Selected {aa_ca_atoms.n_atoms} atoms")
 
aa_positions = []
for ts in u_aa.trajectory:
    aa_positions.append(aa_ca_atoms.positions.copy())
aa_positions = np.array(aa_positions)
 
print(f"  ✓ AA positions extracted")
print(f"    Shape: {aa_positions.shape}")
 
# Extract CG backbone positions
print("\n[2/2] Extracting CG backbone positions...")
cg_ca_atoms = u_cg.select_atoms(cg_rdf_sel)
print(f"  Selected {cg_ca_atoms.n_atoms} atoms")
 
cg_positions = []
for ts in u_cg.trajectory:
    cg_positions.append(cg_ca_atoms.positions.copy())
cg_positions = np.array(cg_positions)
 
print(f"  ✓ CG positions extracted")
print(f"    Shape: {cg_positions.shape}")
 
print(f"\n✓ Position extraction complete")
print("="*70)

## 6. Define Collective Variables

Define physics-informed CVs to evaluate CG model quality.

In [ ]:
print("\n" + "="*70)
print("COLLECTIVE VARIABLE DEFINITIONS")
print("="*70)

def radius_of_gyration(features):
    """
    Radius of gyration from xyz features.
    
    Rg = sqrt(mean((r_i - r_com)^2))
    
    Parameters
    ----------
    features : np.ndarray
        Flattened xyz coordinates, shape (n_frames, n_features)
        where n_features = n_atoms * 3
        
    Returns
    -------
    rg : np.ndarray
        Radius of gyration for each frame, shape (n_frames,)
    """
    n_frames, n_features = features.shape
    n_atoms = n_features // 3
    
    # Reshape to (n_frames, n_atoms, 3)
    positions = features.reshape(n_frames, n_atoms, 3)
    
    # Center of mass
    com = positions.mean(axis=1, keepdims=True)
    
    # Rg calculation
    rg = np.sqrt(np.mean(np.sum((positions - com)**2, axis=2), axis=1))
    
    return rg


def end_to_end_distance(features):
    """
    End-to-end distance (first to last CA).
    
    For Upside (N, CA, C pattern): CA is at indices 1, 4, 7, ...
    
    Parameters
    ----------
    features : np.ndarray
        Flattened xyz coordinates
    Returns
    -------
    e2e : np.ndarray
        End-to-end distance for each frame
    """
    n_frames, n_features = features.shape
    n_atoms = n_features // 3
    
    positions = features.reshape(n_frames, n_atoms, 3)
    
    # For Upside: N, CA, C repeating pattern
    # CA is at indices: 1, 4, 7, 10, ...
    ca_indices = np.arange(1, n_atoms, 3)
    
    if len(ca_indices) < 2:
        raise ValueError("Not enough CA atoms for end-to-end distance")
    
    first_ca = positions[:, ca_indices[0], :]   # First CA
    last_ca = positions[:, ca_indices[-1], :]   # Last CA
    
    e2e = np.linalg.norm(last_ca - first_ca, axis=1)
    
    return e2e


def contact_number(features, cutoff=8.0):
    """
    Number of CA-CA contacts within cutoff distance.
    
    Contact definition: CA-CA distance < cutoff (anstrom)
    Excludes nearest neighbors (i, i+1)
    
    Parameters
    ----------
    features : np.ndarray
        Flattened xyz coordinates
    cutoff : float
        Contact distance cutoff in nm (default: 0.8 nm = 8 Å)
        
    Returns
    -------
    contacts : np.ndarray
        Number of contacts for each frame
    """
    n_frames, n_features = features.shape
    n_atoms = n_features // 3
    
    positions = features.reshape(n_frames, n_atoms, 3)
    
    # Extract CA positions (every 3rd atom starting at index 1)
    ca_indices = np.arange(1, n_atoms, 3)
    ca_positions = positions[:, ca_indices, :]
    
    n_ca = len(ca_indices)
    contacts = np.zeros(n_frames)
    
    for frame_idx in range(n_frames):
        ca_frame = ca_positions[frame_idx]
        
        # Count contacts (exclude neighbors)
        for i in range(n_ca):
            for j in range(i + 2, n_ca):  # Skip i and i+1
                dist = np.linalg.norm(ca_frame[i] - ca_frame[j])
                if dist < cutoff:
                    contacts[frame_idx] += 1
    
    return contacts


# Define CV list
cv_list = [
    ('Rg', radius_of_gyration),
    ('end_to_end', end_to_end_distance),
    ('contacts', contact_number)
]

print(f"\nDefined {len(cv_list)} collective variables:")
for cv_name, cv_func in cv_list:
    print(f"  - {cv_name}")

# Test CVs on CG data
print("\n" + "-"*70)
print("Testing CVs on CG trajectory:")
print("-"*70)
for cv_name, cv_func in cv_list:
    try:
        cv_values = cv_func(cg_features)
        print(f"  {cv_name:15s}: mean={cv_values.mean():7.3f}, std={cv_values.std():7.3f}, "
              f"min={cv_values.min():7.3f}, max={cv_values.max():7.3f}")
    except Exception as e:
        print(f"  {cv_name:15s}: ERROR - {e}")

print("\n✓ CV definitions complete")

## 7. Initialize CGEvaluator

In [ ]:
print("\n" + "="*70)
print("INITIALIZING CG EVALUATOR")
print("="*70)

evaluator = CGEvaluator(
    cg_traj=cg_features,
    aa_traj=aa_features,
    cg_positions=cg_positions,
    aa_positions=aa_positions
)

print("\n✓ Evaluator initialized successfully")

## 8. Run Complete Evaluation

This will compute all metrics across three dimensions:
1. Distribution-based (RDF + CV distributions)
2. Dynamics-based (autocorrelation times)
3. Sampling efficiency (ESS, basin discovery, variance)

In [ ]:
print("\n" + "="*70)
print("RUNNING COMPLETE EVALUATION")
print("="*70)

report = evaluator.evaluate_all(
    cv_list=cv_list,
    rdf_params=RDF_PARAMS,
    n_clusters=N_CLUSTERS,
    output_dir=OUTPUT_DIR,
    save_results=True
)

print("\n✓ Evaluation complete!")

## 9. Generate Summary

In [ ]:
summary = evaluator.generate_summary(report)
print("\n" + summary)

# Save summary to file
summary_file = OUTPUT_DIR / "evaluation_summary.txt"
with open(summary_file, 'w') as f:
    f.write(summary)

print(f"\n✓ Summary saved to: {summary_file}")

## 10. Visualization

### 10.1 RDF Comparison

In [ ]:
if 'rdf' in report['distribution']:
    rdf = report['distribution']['rdf']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # g(r) overlay
    ax1.plot(rdf['r_bins'], rdf['g_cg'], label='CG (Upside)', 
             linewidth=2, color='#2E86AB')
    ax1.plot(rdf['r_bins'], rdf['g_aa'], label='AA (CHARMM)', 
             linewidth=2, color='#A23B72', alpha=0.7)
    ax1.set_xlabel('r (nm)', fontsize=12)
    ax1.set_ylabel('g(r)', fontsize=12)
    ax1.set_title('Radial Distribution Function', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Difference plot
    diff = rdf['g_cg'] - rdf['g_aa']
    ax2.plot(rdf['r_bins'], diff, color='#F18F01', linewidth=2)
    ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax2.fill_between(rdf['r_bins'], 0, diff, alpha=0.3, color='#F18F01')
    ax2.set_xlabel('r (nm)', fontsize=12)
    ax2.set_ylabel('g_CG(r) - g_AA(r)', fontsize=12)
    ax2.set_title(f'Difference (L2 error: {rdf["l2_error"]:.4f}, JS div: {rdf["js_divergence"]:.4f})', 
                  fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'rdf_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ RDF plot saved")
else:
    print("⚠ RDF comparison skipped (positions not provided)")

### 10.2 CV Distribution Comparisons

In [ ]:
n_cvs = len(cv_list)
fig, axes = plt.subplots(n_cvs, 1, figsize=(12, 5*n_cvs))
if n_cvs == 1:
    axes = [axes]

for idx, (cv_name, _) in enumerate(cv_list):
    dist = report['distribution'][cv_name]
    ax = axes[idx]
    
    # Bin centers and width
    bin_centers = 0.5 * (dist['bin_edges'][1:] + dist['bin_edges'][:-1])
    width = bin_centers[1] - bin_centers[0]
    
    # Plot histograms
    ax.bar(bin_centers, dist['cg_hist'], width=width*0.8, 
           label='CG (Upside)', alpha=0.7, color='#2E86AB', edgecolor='black', linewidth=0.5)
    
    if 'aa_hist' in dist:
        ax.bar(bin_centers, dist['aa_hist'], width=width*0.8,
               label='AA (CHARMM)', alpha=0.6, color='#A23B72', edgecolor='black', linewidth=0.5)
        
        # Metrics in title
        title = f"{cv_name} Distribution\n"
        title += f"KS={dist['ks_statistic']:.3f} (p={dist['ks_pvalue']:.3e}), "
        title += f"JS={dist['js_divergence']:.3f}, "
        title += f"Overlap={dist['overlap_coefficient']:.3f}"
        ax.set_title(title, fontsize=13, fontweight='bold')
    else:
        ax.set_title(f"{cv_name} Distribution (CG only)", 
                    fontsize=13, fontweight='bold')
    
    ax.set_xlabel(f'{cv_name}', fontsize=12)
    ax.set_ylabel('Probability Density', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cv_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ CV distribution plots saved")

### 10.3 Autocorrelation Functions

In [ ]:
fig, axes = plt.subplots(n_cvs, 1, figsize=(12, 5*n_cvs))
if n_cvs == 1:
    axes = [axes]

for idx, (cv_name, _) in enumerate(cv_list):
    dyn = report['dynamics'][cv_name]
    ax = axes[idx]
    
    # Plot ACF
    lag = np.arange(len(dyn['acf_cg']))
    ax.plot(lag, dyn['acf_cg'], 
            label=f"CG (τ={dyn['tau_cg']:.1f} frames)", 
            linewidth=2.5, color='#2E86AB')
    
    if 'acf_aa' in dyn:
        lag_aa = np.arange(len(dyn['acf_aa']))
        ax.plot(lag_aa, dyn['acf_aa'], 
                label=f"AA (τ={dyn['tau_aa']:.1f} frames)", 
                linewidth=2.5, color='#A23B72', alpha=0.7)
        
        title = f"{cv_name} Autocorrelation Function\n"
        title += f"Speedup: {dyn['speedup']:.2f}× faster decorrelation in CG"
        ax.set_title(title, fontsize=13, fontweight='bold')
    else:
        ax.set_title(f"{cv_name} Autocorrelation", fontsize=13, fontweight='bold')
    
    ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
    ax.axhline(np.exp(-1), color='gray', linestyle=':', alpha=0.5, linewidth=1, 
               label='e^-1 threshold')
    ax.set_xlabel('Lag (frames)', fontsize=12)
    ax.set_ylabel('ACF', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.2, 1.1])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Autocorrelation plots saved")

### 10.4 Sampling Efficiency Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cv_names = [name for name, _ in cv_list]
colors_cg = '#2E86AB'
colors_aa = '#A23B72'

# ESS comparison
ax = axes[0]
ess_cg = [report['sampling'][name]['ess']['ess_cg'] for name in cv_names]

if 'ess_aa' in report['sampling'][cv_names[0]]['ess']:
    ess_aa = [report['sampling'][name]['ess']['ess_aa'] for name in cv_names]
    
    x = np.arange(len(cv_names))
    width = 0.35
    ax.bar(x - width/2, ess_cg, width, label='CG', alpha=0.8, color=colors_cg)
    ax.bar(x + width/2, ess_aa, width, label='AA', alpha=0.8, color=colors_aa)
    ax.set_xticks(x)
    ax.legend(fontsize=11)
else:
    ax.bar(cv_names, ess_cg, alpha=0.8, color=colors_cg)

ax.set_xticklabels(cv_names, rotation=15, ha='right')
ax.set_ylabel('Effective Sample Size', fontsize=12)
ax.set_title('Effective Sample Size', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Basin coverage
ax = axes[1]
coverage_cg = [report['sampling'][name]['basin']['coverage_cg'] for name in cv_names]

if 'coverage_aa' in report['sampling'][cv_names[0]]['basin']:
    coverage_aa = [report['sampling'][name]['basin']['coverage_aa'] for name in cv_names]
    
    x = np.arange(len(cv_names))
    ax.bar(x - width/2, coverage_cg, width, label='CG', alpha=0.8, color=colors_cg)
    ax.bar(x + width/2, coverage_aa, width, label='AA', alpha=0.8, color=colors_aa)
    ax.set_xticks(x)
    ax.legend(fontsize=11)
else:
    ax.bar(cv_names, coverage_cg, alpha=0.8, color=colors_cg)

ax.set_xticklabels(cv_names, rotation=15, ha='right')
ax.set_ylabel('Basin Coverage Ratio', fontsize=12)
ax.set_title('State Space Exploration', fontsize=13, fontweight='bold')
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# Variance comparison
ax = axes[2]
var_cg = [report['sampling'][name]['variance']['variance_cg'] for name in cv_names]

if 'variance_aa' in report['sampling'][cv_names[0]]['variance']:
    var_aa = [report['sampling'][name]['variance']['variance_aa'] for name in cv_names]
    
    x = np.arange(len(cv_names))
    ax.bar(x - width/2, var_cg, width, label='CG', alpha=0.8, color=colors_cg)
    ax.bar(x + width/2, var_aa, width, label='AA', alpha=0.8, color=colors_aa)
    ax.set_xticks(x)
    ax.legend(fontsize=11)
else:
    ax.bar(cv_names, var_cg, alpha=0.8, color=colors_cg)

ax.set_xticklabels(cv_names, rotation=15, ha='right')
ax.set_ylabel('Variance', fontsize=12)
ax.set_title('CV Space Variance', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sampling_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Sampling efficiency plots saved")

## 11. Final Summary

In [ ]:
print("\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)

print(f"\n📊 Results saved to: {OUTPUT_DIR}")
print("\n📁 Generated files:")
print("  ✓ evaluation_results.pkl      - Complete results (Python pickle)")
print("  ✓ evaluation_summary.txt       - Text summary")
print("  ✓ alignment_rmsd.png           - Alignment quality check")
print("  ✓ rdf_comparison.png           - RDF analysis")
print("  ✓ cv_distributions.png         - CV distribution comparisons")
print("  ✓ autocorrelation.png          - Dynamics analysis")
print("  ✓ sampling_summary.png         - Sampling efficiency metrics")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("""
1. Review the evaluation_summary.txt for key metrics
2. Examine the plots to understand CG model performance
3. Use these metrics to benchmark different CG force fields
4. Iterate on CG model parameters to improve performance

This evaluation provides the quality standard for BEAM CG models.
""")

## 12. Key Metrics Summary Table

In [ ]:
!pip install pandas
import pandas as pd

# Create summary table
summary_data = []

for cv_name in cv_names:
    row = {'CV': cv_name}
    
    # Distribution
    dist = report['distribution'][cv_name]
    if 'ks_statistic' in dist:
        row['KS_stat'] = f"{dist['ks_statistic']:.3f}"
        row['JS_div'] = f"{dist['js_divergence']:.3f}"
        row['Overlap'] = f"{dist['overlap_coefficient']:.3f}"
    
    # Dynamics
    dyn = report['dynamics'][cv_name]
    row['tau_CG'] = f"{dyn['tau_cg']:.1f}"
    if 'tau_aa' in dyn:
        row['tau_AA'] = f"{dyn['tau_aa']:.1f}"
        row['Speedup'] = f"{dyn['speedup']:.2f}×"
    
    # Sampling
    samp = report['sampling'][cv_name]
    row['ESS_CG'] = f"{samp['ess']['ess_cg']:.0f}"
    row['Basin_Cov'] = f"{samp['basin']['coverage_cg']:.2f}"
    row['Variance'] = f"{samp['variance']['variance_cg']:.4f}"
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("KEY METRICS SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv(OUTPUT_DIR / 'metrics_summary.csv', index=False)
print(f"\n✓ Metrics table saved to: {OUTPUT_DIR / 'metrics_summary.csv'}")

print("\n" + "="*70)
print("✅ STAGE 1 CG EVALUATION COMPLETE!")
print("="*70)